# torch-batteries overhead over vanilla PyTorch

This notebook measures the wall-clock time `torch-batteries` adds over hand-written PyTorch training and evaluation loops in a typical setup: a progress bar with running loss and accuracy, and background data-loading workers.
Every loop runs the same ResNet on the same synthetic batches with the same optimizer settings. PyTorch Lightning, PyTorch Ignite, fastai, and Keras (PyTorch backend) can optionally be measured alongside.

[Training guide](https://michalszc.github.io/torch-batteries/guides/training/) · [Events API](https://michalszc.github.io/torch-batteries/reference/events/)

## 0. Prerequisites

The next cell installs only what is missing, so a local environment keeps its own install. The `INCLUDE_*` flags choose which other frameworks to benchmark.

A CUDA GPU is strongly recommended. Nothing is downloaded.

In [1]:
import importlib.util

INCLUDE_LIGHTNING = True
INCLUDE_IGNITE = True
INCLUDE_FASTAI = True
INCLUDE_KERAS = True

# Keep existing installs; otherwise install from PyPI.
packages = []
if importlib.util.find_spec("torch_batteries") is None:
    packages.append("torch-batteries")
if INCLUDE_LIGHTNING and importlib.util.find_spec("lightning") is None:
    packages.append("lightning")
if INCLUDE_IGNITE and importlib.util.find_spec("ignite") is None:
    packages.append("pytorch-ignite")
if INCLUDE_FASTAI and importlib.util.find_spec("fastai") is None:
    packages.append("fastai")
if INCLUDE_KERAS and importlib.util.find_spec("keras") is None:
    packages.append("keras")
if packages:
    %pip install {" ".join(packages)}

## 1. Configuration

- `WARMUP_EPOCHS` absorb CUDA context creation, cuDNN autotuning, and allocator growth, and are not timed.
- Each of the `TIMED_EPOCHS` is a separate one-epoch call per loop. The loops are interleaved with a rotating start, so GPU clock and thermal drift affect all of them equally.
- Workers are persistent, so their start-up is paid once rather than in every timed call.

In [ ]:
import contextlib
import gc
import logging
import os
import sys
import time
import warnings
from functools import partial

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch import nn
from torch.nn import functional as nn_functional
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import models
from tqdm import tqdm

import torch_batteries
from torch_batteries import Battery, Event, EventContext, StepOutput, charge
from torch_batteries.utils.device import get_device

# Models and data shape
MODEL_NAMES = ("resnet18",)
NUM_CLASSES = 10
IMAGE_SIZE = 224
BATCH_SIZE = 64
BATCHES_PER_EPOCH = 50
SAMPLE_POOL_SIZE = 256

# Timing protocol
WARMUP_EPOCHS = 1
TIMED_EPOCHS = 10
SINGLE_BATCH_REPEATS = 50
INCLUDE_EVALUATION = True
 
# Statistics
BOOTSTRAP_RESAMPLES = 10_000
CONFIDENCE_LEVEL = 0.95

# Training and loader settings shared by all loops
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_WORKERS = 2
CUDNN_BENCHMARK = True
SEED = 42

LOOP_LABELS = {
    "vanilla": "vanilla PyTorch",
    "battery": "torch-batteries",
    "lightning": "Lightning",
    "ignite": "Ignite",
    "fastai": "fastai",
    "keras": "Keras",
}
LOOPS = ("vanilla", "battery") + tuple(
    name
    for name, included in (
        ("lightning", INCLUDE_LIGHTNING),
        ("ignite", INCLUDE_IGNITE),
        ("fastai", INCLUDE_FASTAI),
        ("keras", INCLUDE_KERAS),
    )
    if included
)

DEVICE = get_device("auto")  # or "cuda", "mps", "cpu"
PIN_MEMORY = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = CUDNN_BENCHMARK

print(f"torch-batteries: {torch_batteries.__version__}")
print(f"torch: {torch.__version__}")

if INCLUDE_LIGHTNING:
    import lightning

    logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
    warnings.filterwarnings("ignore", ".*does not have many workers.*")
    print(f"lightning: {lightning.__version__}")

if INCLUDE_IGNITE:
    import ignite

    print(f"ignite: {ignite.__version__}")

if INCLUDE_FASTAI:
    import fastai.callback.progress
    import fastprogress.fastprogress

    # Plain-text bars like the other loops instead of notebook HTML.
    fastai.callback.progress.master_bar, fastai.callback.progress.progress_bar = (
        fastprogress.fastprogress.force_console_behavior()
    )
    warnings.filterwarnings("ignore", "Your generator is empty")
    print(f"fastai: {fastai.__version__}")

if INCLUDE_KERAS:
    os.environ["KERAS_BACKEND"] = "torch"  # must be set before Keras is imported
    import keras

    print(f"keras: {keras.__version__} ({keras.backend.backend()} backend)")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

torch-batteries: 1.0.0
torch: 2.9.1+cu128
lightning: 2.6.6
ignite: 0.5.5
fastai: 2.8.12
keras: 3.15.1 (torch backend)
Device: cuda
GPU: NVIDIA GeForce GTX 1660 Ti


## 2. Synthetic data

A fixed pool of random images is recycled, so all loops read identical batches with no download, decoding, or augmentation, and longer epochs need no more memory.

In [3]:
class SyntheticImages(Dataset):
    def __init__(self, length, pool_size, image_size, num_classes, seed):
        generator = torch.Generator().manual_seed(seed)
        self.length = length
        self.images = torch.randn(
            pool_size, 3, image_size, image_size, generator=generator
        )
        self.labels = torch.randint(0, num_classes, (pool_size,), generator=generator)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        index %= len(self.labels)
        return self.images[index], self.labels[index]


def make_loader(
    batches,
    batch_size=BATCH_SIZE,
    pool_size=SAMPLE_POOL_SIZE,
    image_size=IMAGE_SIZE,
    num_classes=NUM_CLASSES,
    seed=SEED,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
):
    dataset = SyntheticImages(
        length := batches * batch_size,
        min(pool_size, length),
        image_size,
        num_classes,
        seed,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
        pin_memory=pin_memory,
    )


epoch_loader = make_loader(BATCHES_PER_EPOCH)
single_batch_loader = make_loader(1)
print(
    f"Epoch: {len(epoch_loader)} batches of "
    f"{BATCH_SIZE} × 3 × {IMAGE_SIZE} × {IMAGE_SIZE}"
)

Epoch: 50 batches of 64 × 3 × 224 × 224


## 3. The loops

Each loop does the same bookkeeping in its framework's idiomatic way, and evaluates through the framework's standard entry point. Choices that are not obvious from the code:

- **Lightning** builds a fresh `Trainer` per call, as `fit` is normally used, without logger, checkpointing, or model summary, since torch-batteries writes none of these by default.
- **fastai** gets an empty validation loader during training because `fit` always validates.
- **Keras** must use its own `SGD`, which applies the same update as `torch.optim.SGD` but is a different implementation, so its cost is part of the measurement. Keras also leaves batches from a PyTorch `DataLoader` on the CPU, hence the `train_step` and `test_step` overrides.
- Lightning, Ignite, and fastai default to widget or HTML bars in notebooks; they are switched to plain text so that `hidden_output` can hide them.

In [4]:
def batch_accuracy(predictions, targets):
    return (predictions.argmax(dim=1) == targets).float().mean()


class BenchmarkResNet(nn.Module):
    def __init__(self, model_name, num_classes=NUM_CLASSES):
        super().__init__()
        self.network = getattr(models, model_name)(num_classes=num_classes)

    def forward(self, images):
        return self.network(images)

    @charge(Event.TRAIN_STEP)
    @charge(Event.VALIDATION_STEP)
    def step(self, context: EventContext):
        images, targets = context["batch"]
        logits = self(images)
        return StepOutput(
            loss=nn_functional.cross_entropy(logits, targets),
            predictions=logits,
            targets=targets,
        )


def make_optimizer(model, lr=LEARNING_RATE, momentum=MOMENTUM):
    return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

In [5]:
def vanilla_train_epoch(
    model, optimizer, loader, device=DEVICE, desc=LOOP_LABELS["vanilla"]
):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    with tqdm(loader, desc=desc) as progress:
        for images, targets in progress:
            images = images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = nn_functional.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step()
            batch_size = len(targets)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_samples += batch_size
            progress.set_postfix_str(
                f"Loss={total_loss / total_samples:.4f}, "
                f"Accuracy={total_correct / total_samples:.4f}"
            )
    return total_loss / total_samples


def vanilla_evaluate_epoch(
    model, loader, device=DEVICE, desc=f"{LOOP_LABELS['vanilla']} evaluation"
):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    with (
        torch.no_grad(),
        tqdm(loader, desc=desc) as progress,
    ):
        for images, targets in progress:
            images = images.to(device)
            targets = targets.to(device)
            logits = model(images)
            loss = nn_functional.cross_entropy(logits, targets)
            batch_size = len(targets)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_samples += batch_size
            progress.set_postfix_str(
                f"Loss={total_loss / total_samples:.4f}, "
                f"Accuracy={total_correct / total_samples:.4f}"
            )
    return total_loss / total_samples

In [6]:
def battery_train_epoch(battery, loader):
    return battery.train(loader, epochs=1, verbose=1)


def battery_evaluate_epoch(battery, loader):
    return battery.validate(loader, verbose=1)

In [7]:
if INCLUDE_LIGHTNING:

    class LightningResNet(lightning.LightningModule):
        def __init__(self, model_name, num_classes=NUM_CLASSES):
            super().__init__()
            self.network = getattr(models, model_name)(num_classes=num_classes)
            self.optimizer = make_optimizer(self)

        def forward(self, images):
            return self.network(images)

        def training_step(self, batch, batch_idx):
            images, targets = batch
            logits = self(images)
            loss = nn_functional.cross_entropy(logits, targets)
            self.log("loss", loss, on_step=True, on_epoch=True, prog_bar=True)
            self.log(
                "acc",
                batch_accuracy(logits, targets),
                on_step=True,
                on_epoch=True,
                prog_bar=True,
            )
            return loss

        def validation_step(self, batch, batch_idx):
            images, targets = batch
            logits = self(images)
            loss = nn_functional.cross_entropy(logits, targets)
            self.log("loss", loss, prog_bar=True)
            self.log("acc", batch_accuracy(logits, targets), prog_bar=True)

        def configure_optimizers(self):
            return self.optimizer

    class TextProgressBar(lightning.pytorch.callbacks.TQDMProgressBar):
        """Render plain-text bars like the other loops instead of notebook widgets."""

        def init_train_tqdm(self):
            return tqdm(desc=self.train_description, leave=True)

        def init_validation_tqdm(self):
            return tqdm(desc=self.validation_description, leave=True)

    def make_lightning_trainer(device=DEVICE):
        return lightning.Trainer(
            accelerator=device.type,
            devices=1,
            max_epochs=1,
            logger=False,
            enable_checkpointing=False,
            enable_model_summary=False,
            callbacks=[TextProgressBar()],
        )

    def lightning_train_epoch(module, loader, device=DEVICE):
        make_lightning_trainer(device).fit(module, loader)

    def lightning_evaluate_epoch(module, loader, device=DEVICE):
        # verbose=False drops the results table printed after validation.
        make_lightning_trainer(device).validate(module, loader, verbose=False)

In [8]:
if INCLUDE_IGNITE:

    class TextIgniteProgressBar(ignite.handlers.ProgressBar):
        """Render plain-text bars like the other loops instead of notebook widgets."""

        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.pbar_cls = tqdm

    def make_ignite_trainer(
        model, optimizer, device=DEVICE, desc=LOOP_LABELS["ignite"]
    ):
        trainer = ignite.engine.create_supervised_trainer(
            model,
            optimizer,
            loss_fn=nn_functional.cross_entropy,
            device=device,
            output_transform=lambda _images, targets, logits, loss: (
                logits,
                targets,
                loss.item(),
            ),
        )
        ignite.metrics.RunningAverage(output_transform=lambda output: output[2]).attach(
            trainer, "loss"
        )
        ignite.metrics.RunningAverage(
            ignite.metrics.Accuracy(output_transform=lambda output: output[:2])
        ).attach(trainer, "acc")
        TextIgniteProgressBar(persist=True, desc=desc).attach(
            trainer, metric_names="all"
        )
        return trainer

    def make_ignite_evaluator(
        model, device=DEVICE, desc=f"{LOOP_LABELS['ignite']} evaluation"
    ):
        evaluator = ignite.engine.create_supervised_evaluator(
            model,
            metrics={
                "loss": ignite.metrics.Loss(nn_functional.cross_entropy),
                "acc": ignite.metrics.Accuracy(),
            },
            device=device,
        )
        TextIgniteProgressBar(persist=True, desc=desc).attach(evaluator)
        return evaluator

    def ignite_train_epoch(trainer, loader):
        trainer.run(loader, max_epochs=1)

    def ignite_evaluate_epoch(evaluator, loader):
        evaluator.run(loader)

In [9]:
if INCLUDE_FASTAI:
    no_validation_loader = DataLoader(TensorDataset(torch.empty(0)))

    def make_fastai_learner(model, momentum=MOMENTUM):
        learner = fastai.learner.Learner(
            # Each call below passes the loader it runs on.
            fastai.data.core.DataLoaders(no_validation_loader, no_validation_loader),
            model,
            loss_func=nn_functional.cross_entropy,
            opt_func=partial(
                fastai.optimizer.OptimWrapper,
                opt=partial(torch.optim.SGD, momentum=momentum),
            ),
            metrics=fastai.metrics.accuracy,
        )
        learner.recorder.train_metrics = True
        return learner

    def fastai_train_epoch(learner, loader, lr=LEARNING_RATE):
        learner.dls = fastai.data.core.DataLoaders(loader, no_validation_loader)
        learner.fit(1, lr=lr)

    def fastai_evaluate_epoch(learner, loader):
        learner.validate(dl=loader)

In [10]:
if INCLUDE_KERAS:

    class KerasResNet(keras.Model):
        def __init__(self, module, device=DEVICE):
            super().__init__()
            self.batch_device = device
            self.network = keras.layers.TorchModuleWrapper(module)

        def call(self, images):
            return self.network(images)

        def train_step(self, data):
            # Keras leaves batches from a PyTorch DataLoader on the CPU.
            images, targets = data
            return super().train_step(
                (images.to(self.batch_device), targets.to(self.batch_device))
            )

        def test_step(self, data):
            images, targets = data
            return super().test_step(
                (images.to(self.batch_device), targets.to(self.batch_device))
            )

    def make_keras_model(module, lr=LEARNING_RATE, momentum=MOMENTUM, device=DEVICE):
        model = KerasResNet(module, device)
        model.compile(
            optimizer=keras.optimizers.SGD(learning_rate=lr, momentum=momentum),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
        return model

    def keras_train_epoch(model, loader):
        model.fit(loader, epochs=1, shuffle=False, verbose=1)

    def keras_evaluate_epoch(model, loader):
        model.evaluate(loader, verbose=1)

In [11]:
def synchronize(device=DEVICE):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


@contextlib.contextmanager
def hidden_output():
    """Send framework progress bars and logs to os.devnull instead of the notebook."""
    with (
        open(os.devnull, "w") as sink,
        contextlib.redirect_stdout(sink),
        contextlib.redirect_stderr(sink),
    ):
        yield


def timed(run, device=DEVICE):
    synchronize(device)
    start = time.perf_counter()
    run()
    synchronize(device)
    return time.perf_counter() - start

## 4. Benchmark protocol

All models start from identical weights. With every timed epoch all loops are run back to back, so each epoch is a paired comparison with the vanilla loop under the same conditions. The single-batch calls get one untimed round first, because their loader starts its workers on the first call.

Training and evaluation are benchmarked separately, each on fresh models. Set `INCLUDE_EVALUATION = False` to skip evaluation.

In [12]:
def make_models(model_name, loops=LOOPS, seed=SEED, device=DEVICE):
    """Build one model per loop, all starting from identical weights."""
    torch.manual_seed(seed)
    weights = BenchmarkResNet(model_name).state_dict()
    models_by_loop = {}
    for name in loops:
        model_class = LightningResNet if name == "lightning" else BenchmarkResNet
        model = model_class(model_name)
        model.load_state_dict(weights)
        models_by_loop[name] = model.to(device)
    return models_by_loop


def make_training_runs(models_by_loop, device=DEVICE):
    """Return a callable per loop that trains its model for one epoch on a loader."""
    vanilla_model = models_by_loop["vanilla"]
    battery_model = models_by_loop["battery"]
    battery = Battery(
        battery_model,
        device=device,
        optimizer=make_optimizer(battery_model),
        metrics={"accuracy": batch_accuracy},
    )
    runs = {
        "vanilla": partial(
            vanilla_train_epoch, vanilla_model, make_optimizer(vanilla_model)
        ),
        "battery": partial(battery_train_epoch, battery),
    }
    if "lightning" in models_by_loop:
        runs["lightning"] = partial(lightning_train_epoch, models_by_loop["lightning"])
    if "ignite" in models_by_loop:
        ignite_model = models_by_loop["ignite"]
        trainer = make_ignite_trainer(ignite_model, make_optimizer(ignite_model))
        runs["ignite"] = partial(ignite_train_epoch, trainer)
    if "fastai" in models_by_loop:
        learner = make_fastai_learner(models_by_loop["fastai"])
        runs["fastai"] = partial(fastai_train_epoch, learner)
    if "keras" in models_by_loop:
        keras_model = make_keras_model(models_by_loop["keras"])
        runs["keras"] = partial(keras_train_epoch, keras_model)
    return runs


def make_evaluation_runs(models_by_loop, device=DEVICE):
    """Return a callable per loop that evaluates its model for one epoch on a loader."""
    battery = Battery(
        models_by_loop["battery"], device=device, metrics={"accuracy": batch_accuracy}
    )
    runs = {
        "vanilla": partial(vanilla_evaluate_epoch, models_by_loop["vanilla"]),
        "battery": partial(battery_evaluate_epoch, battery),
    }
    if "lightning" in models_by_loop:
        runs["lightning"] = partial(
            lightning_evaluate_epoch, models_by_loop["lightning"]
        )
    if "ignite" in models_by_loop:
        evaluator = make_ignite_evaluator(models_by_loop["ignite"])
        runs["ignite"] = partial(ignite_evaluate_epoch, evaluator)
    if "fastai" in models_by_loop:
        learner = make_fastai_learner(models_by_loop["fastai"])
        runs["fastai"] = partial(fastai_evaluate_epoch, learner)
    if "keras" in models_by_loop:
        keras_model = make_keras_model(models_by_loop["keras"])
        runs["keras"] = partial(keras_evaluate_epoch, keras_model)
    return runs


def interleaved(runs, repeats, progress, device=DEVICE):
    samples = {name: [] for name in runs}
    names = list(runs)
    for repeat in range(repeats):
        shift = repeat % len(names)
        for name in names[shift:] + names[:shift]:
            samples[name].append(timed(runs[name], device))
            progress.update()
    return samples


def benchmark(
    model_name,
    mode,
    make_runs,
    epoch_loader,
    single_batch_loader,
    loops=LOOPS,
    warmup_epochs=WARMUP_EPOCHS,
    timed_epochs=TIMED_EPOCHS,
    single_batch_repeats=SINGLE_BATCH_REPEATS,
    seed=SEED,
    device=DEVICE,
):
    """Time every loop on full epochs and on single-batch calls."""
    runs = make_runs(make_models(model_name, loops, seed, device), device)
    epoch_runs = {name: partial(run, epoch_loader) for name, run in runs.items()}
    single_runs = {
        name: partial(run, single_batch_loader) for name, run in runs.items()
    }
    notebook_output = sys.stderr  # the bars keep writing here while output is hidden
    with hidden_output():
        with tqdm(
            total=len(runs) * (warmup_epochs + timed_epochs),
            unit="epoch",
            file=notebook_output,
        ) as progress:
            progress.set_description(f"{model_name} {mode}: warm-up epochs")
            interleaved(epoch_runs, warmup_epochs, progress, device)
            progress.set_description(f"{model_name} {mode}: timed epochs")
            epoch_samples = interleaved(epoch_runs, timed_epochs, progress, device)
        with tqdm(
            total=len(runs) * (1 + single_batch_repeats),
            unit="call",
            file=notebook_output,
        ) as progress:
            # The single-batch loader starts its workers on the first call.
            progress.set_description(f"{model_name} {mode}: single-batch warm-up")
            interleaved(single_runs, 1, progress, device)
            progress.set_description(f"{model_name} {mode}: single batches")
            single_samples = interleaved(
                single_runs, single_batch_repeats, progress, device
            )
    del runs, epoch_runs, single_runs
    gc.collect()  # framework objects hold reference cycles that keep models alive
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"epochs": epoch_samples, "single-batch calls": single_samples}


def bootstrap_median_ci(
    values, rng, resamples=BOOTSTRAP_RESAMPLES, confidence=CONFIDENCE_LEVEL
):
    medians = np.median(rng.choice(values, size=(resamples, len(values))), axis=1)
    tail = 100 * (1 - confidence) / 2
    return np.percentile(medians, [tail, 100 - tail])


def summarize(
    samples,
    slower_label,
    labels=LOOP_LABELS,
    resamples=BOOTSTRAP_RESAMPLES,
    confidence=CONFIDENCE_LEVEL,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    vanilla = np.array(samples["vanilla"])
    confidence_label = f"{100 * confidence:g}%"
    rows = {}
    for name, values in samples.items():
        times = np.array(values)
        overheads = 100 * (times - vanilla) / vanilla
        low, high = (
            bootstrap_median_ci(overheads, rng, resamples, confidence)
            if name != "vanilla"
            else (0.0, 0.0)
        )
        rows[labels[name]] = {
            "median [s]": np.median(times),
            "std [s]": np.std(times, ddof=1),
            "median overhead [%]": np.median(overheads),
            f"CI {confidence_label} low [%]": low,
            f"CI {confidence_label} high [%]": high,
            slower_label: f"{np.sum(times > vanilla)}/{len(times)}",
        }
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("loop")

## 5. Training benchmark

In [13]:
samples = {}
for model_name in MODEL_NAMES:
    for kind, kind_samples in benchmark(
        model_name, "training", make_training_runs, epoch_loader, single_batch_loader
    ).items():
        samples[model_name, "training", kind] = kind_samples

resnet18 training: single batches: 100%|██████████| 306/306 [01:40<00:00,  3.04call/s]    


## 6. Evaluation benchmark

In [14]:
if INCLUDE_EVALUATION:
    for model_name in MODEL_NAMES:
        for kind, kind_samples in benchmark(
            model_name,
            "evaluation",
            make_evaluation_runs,
            epoch_loader,
            single_batch_loader,
        ).items():
            samples[model_name, "evaluation", kind] = kind_samples

resnet18 evaluation: single batches: 100%|██████████| 306/306 [00:55<00:00,  5.47call/s]    


## 7. Results

In [15]:
slower_labels = {"epochs": "slower epochs", "single-batch calls": "slower batches"}
results = {
    key: summarize(kind_samples, slower_labels[key[2]])
    for key, kind_samples in samples.items()
}
for (model_name, mode, kind), table in results.items():
    display(Markdown(f"### {model_name} {mode}: {kind}"), table.round(4))

### resnet18 training: epochs

,median [s],std [s],median overhead [%],CI 95% low [%],CI 95% high [%],slower epochs
loop,,,,,,
vanilla PyTorch,9.4147,0.0482,0.0000,0.0000,0.0000,0/10
torch-batteries,9.3670,0.0524,-0.6765,-0.8840,0.0728,3/10
Lightning,9.6939,0.0419,2.9739,2.5860,3.4908,10/10
Ignite,9.4307,0.0688,0.0891,-0.4248,0.7869,6/10
fastai,9.3549,0.0273,-0.7770,-0.8735,-0.1448,1/10
Keras,9.5360,0.0320,1.0852,0.9749,1.6158,10/10


### resnet18 training: single-batch calls

,median [s],std [s],median overhead [%],CI 95% low [%],CI 95% high [%],slower batches
loop,,,,,,
vanilla PyTorch,0.2803,0.0159,0.0000,0.0000,0.0000,0/50
torch-batteries,0.2402,0.0171,-14.2492,-14.6670,-12.7182,12/50
Lightning,0.6456,0.0128,131.9634,129.3710,133.9763,50/50
Ignite,0.3012,0.0327,7.3772,6.8002,8.1058,42/50
fastai,0.2441,0.0223,-12.5892,-13.2845,-12.0193,15/50
Keras,0.2487,0.0031,-10.9795,-11.3647,-10.5128,8/50


### resnet18 evaluation: epochs

,median [s],std [s],median overhead [%],CI 95% low [%],CI 95% high [%],slower epochs
loop,,,,,,
vanilla PyTorch,3.2788,0.0287,0.0000,0.0000,0.0000,0/10
torch-batteries,3.1788,0.0410,-2.6081,-3.5968,-0.7626,0/10
Lightning,3.3920,0.0179,3.3624,2.9319,4.0731,10/10
Ignite,3.2599,0.0239,-0.7343,-1.2656,-0.5653,1/10
fastai,3.2224,0.0196,-1.8677,-2.3310,-1.0682,1/10
Keras,3.5042,0.0244,7.0108,6.6352,7.5947,10/10


### resnet18 evaluation: single-batch calls

,median [s],std [s],median overhead [%],CI 95% low [%],CI 95% high [%],slower batches
loop,,,,,,
vanilla PyTorch,0.1576,0.0166,0.0000,0.0000,0.0000,0/50
torch-batteries,0.1136,0.0172,-27.7951,-28.4718,-25.8258,5/50
Lightning,0.3884,0.0107,147.4297,143.0447,149.4429,50/50
Ignite,0.1775,0.0302,12.5708,10.9965,13.9743,42/50
fastai,0.1196,0.0238,-23.7289,-25.2209,-22.0170,15/50
Keras,0.1274,0.0028,-19.5151,-20.3166,-17.7242,8/50


## 8. Individual epoch times

In [16]:
for (model_name, mode, kind), kind_samples in samples.items():
    if kind != "epochs":
        continue
    epoch_times = pd.DataFrame(
        {LOOP_LABELS[name]: values for name, values in kind_samples.items()},
        index=pd.RangeIndex(1, len(kind_samples["vanilla"]) + 1, name="epoch"),
    )
    display(Markdown(f"### {model_name} {mode} (s)"), epoch_times.round(3))

### resnet18 training (s)

,vanilla PyTorch,torch-batteries,Lightning,Ignite,fastai,Keras
epoch,,,,,,
1,9.367,9.305,9.654,9.382,9.354,9.493
2,9.400,9.471,9.741,9.606,9.314,9.581
3,9.413,9.330,9.656,9.406,9.337,9.514
4,9.420,9.355,9.749,9.347,9.341,9.490
5,9.409,9.378,9.676,9.426,9.402,9.532
6,9.416,9.338,9.688,9.435,9.356,9.518
7,9.311,9.355,9.700,9.441,9.339,9.547
8,9.448,9.455,9.677,9.450,9.368,9.540
9,9.466,9.382,9.701,9.448,9.395,9.569


### resnet18 evaluation (s)

,vanilla PyTorch,torch-batteries,Lightning,Ignite,fastai,Keras
epoch,,,,,,
1,3.267,3.186,3.363,3.254,3.191,3.485
2,3.286,3.265,3.420,3.265,3.227,3.518
3,3.271,3.182,3.394,3.247,3.196,3.498
4,3.263,3.192,3.397,3.200,3.221,3.498
5,3.284,3.166,3.374,3.259,3.262,3.486
6,3.297,3.176,3.406,3.273,3.215,3.562
7,3.197,3.173,3.390,3.260,3.224,3.530
8,3.293,3.276,3.405,3.269,3.210,3.528
9,3.290,3.166,3.388,3.263,3.227,3.510


## 9. Reading the results

- **Median overhead** of the epoch table is the headline number.
- **The confidence interval** containing zero means the overhead is indistinguishable from noise.
- **The single-batch table** mostly shows fixed setup paid once per training or evaluation call.